In [61]:
import boto3

In [62]:
modelId = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# modelId = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"

Below is a message which requires tool use

In [63]:
messages = [{
            "role": "user",
            "content": [{"text": "I have 100€ and put in a bank account with 4% interest per year. How much money do I have after 2 years and 3 months?"}]
        }]

This one does not require tool use

In [53]:
messages = [{
            "role": "user",
            "content": [{"text": "What is the capital of france?"}]
        }]

Below we define a simple tool which allows to perform calculations.

In [64]:
tool_config = {
        "tools": [
            {
                "toolSpec": {
                    "name": "eval_math",
                    "description": "Evaluates the provided mathematical expression.",
                    "inputSchema": {
                        "json": {
                            "type": "object",
                            "properties": {
                                "expr": {
                                    "type": "string",
                                    "description": "The mathematical expression to evaluate. The expression uses Python syntax."
                                }
                            },
                            "required": ["expr"]
                        }
                    }
                }
            }
        ]
    }

In [65]:
client = boto3.client("bedrock-runtime", region_name="us-east-1")  

In [66]:
response = client.converse(modelId=modelId, messages=messages, toolConfig=tool_config)
print(response)

{'ResponseMetadata': {'RequestId': '30d2de6f-71b0-4575-aade-15119646f9d8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 12 Mar 2026 08:07:34 GMT', 'content-type': 'application/json', 'content-length': '449', 'connection': 'keep-alive', 'x-amzn-requestid': '30d2de6f-71b0-4575-aade-15119646f9d8'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'toolUse': {'toolUseId': 'tooluse_w0EqWEWe7WpXog9BOpvEOX', 'name': 'eval_math', 'input': {'expr': '100 * (1 + 0.04) ** (2 + 3/12)'}, 'type': 'tool_use'}}]}}, 'stopReason': 'tool_use', 'usage': {'inputTokens': 611, 'outputTokens': 74, 'totalTokens': 685, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}, 'metrics': {'latencyMs': 1707}}


In the response processing, we have to check whether the model requested a tool use...

Based on https://docs.aws.amazon.com/bedrock/latest/userguide/tool-use.html

In [67]:
output_message = response['output']['message']
messages.append(output_message)

In [68]:
stop_reason = response['stopReason']
print(stop_reason)

tool_use


In [69]:
if stop_reason == 'tool_use':
    # Tool use requested. Call the tool and send the result to the model.
    tool_requests = response['output']['message']['content']
    print(tool_requests)

[{'toolUse': {'toolUseId': 'tooluse_w0EqWEWe7WpXog9BOpvEOX', 'name': 'eval_math', 'input': {'expr': '100 * (1 + 0.04) ** (2 + 3/12)'}, 'type': 'tool_use'}}]


In [70]:
    for tool_request in tool_requests:
        if 'toolUse' in tool_request:
            tool = tool_request['toolUse']
            print("Requesting tool %s. Request: %s", tool['name'], tool['toolUseId'])
            if tool['name'] == 'eval_math':
                tool_result = {}
                inp = tool['input']['expr']
                res = eval(inp)
                print("res = ", res)
                tool_result = {"toolUseId": tool['toolUseId'],
                                       "content": [{"json": {"value": res}}]}
                print(tool_result)
                tool_result_message = {"role": "user", "content": [{"toolResult": tool_result}]}
                messages.append(tool_result_message)
                print(messages)
                # Send the tool result to the model.
                response = client.converse(modelId=modelId,
                                           messages=messages,
                                           toolConfig=tool_config)

                output_message = response['output']['message']

Requesting tool %s. Request: %s eval_math tooluse_w0EqWEWe7WpXog9BOpvEOX
res =  109.22574445233649
{'toolUseId': 'tooluse_w0EqWEWe7WpXog9BOpvEOX', 'content': [{'json': {'value': 109.22574445233649}}]}
[{'role': 'user', 'content': [{'text': 'I have 100€ and put in a bank account with 4% interest per year. How much money do I have after 2 years and 3 months?'}]}, {'role': 'assistant', 'content': [{'toolUse': {'toolUseId': 'tooluse_w0EqWEWe7WpXog9BOpvEOX', 'name': 'eval_math', 'input': {'expr': '100 * (1 + 0.04) ** (2 + 3/12)'}, 'type': 'tool_use'}}]}, {'role': 'user', 'content': [{'toolResult': {'toolUseId': 'tooluse_w0EqWEWe7WpXog9BOpvEOX', 'content': [{'json': {'value': 109.22574445233649}}]}}]}]


In [71]:
print(output_message)

{'role': 'assistant', 'content': [{'text': "After 2 years and 3 months with a 4% annual interest rate, you would have approximately **€109.23** in your bank account.\n\nHere's how this works:\n- Time period: 2 years and 3 months = 2.25 years\n- Formula: A = P(1 + r)^t\n  - P = €100 (principal)\n  - r = 0.04 (4% annual interest)\n  - t = 2.25 years\n- Result: €100 × (1.04)^2.25 ≈ **€109.23**\n\nThis assumes the interest is compounded annually. If your bank compounds interest monthly or daily, the result would be slightly different."}]}


In [72]:
print(output_message["content"][0]["text"])

After 2 years and 3 months with a 4% annual interest rate, you would have approximately **€109.23** in your bank account.

Here's how this works:
- Time period: 2 years and 3 months = 2.25 years
- Formula: A = P(1 + r)^t
  - P = €100 (principal)
  - r = 0.04 (4% annual interest)
  - t = 2.25 years
- Result: €100 × (1.04)^2.25 ≈ **€109.23**

This assumes the interest is compounded annually. If your bank compounds interest monthly or daily, the result would be slightly different.
